# 10. Оценка через EvaluationOrchestrator (без разметки)

**Цель:** показать `EvaluationCase`/`EvaluationOrchestrator` на одном неразмеченном многоканальном случае. Kuramoto-файл не содержит истинного времени события, поэтому кейс помечен `has_truth=False` и не участвует в метриках качества — оркестратор лишь исполняет pipeline и логирует прогресс.

In [1]:
from pathlib import Path
import pandas as pd
from graph_evt_agent import EVTConfig, EvaluationCase, EvaluationOrchestrator, GraphConfig, GraphEVTPipeline, InputConfig, load_timeseries

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
kuramoto_input = load_timeseries(DATA)

In [2]:
BASELINE_SIZE = 300
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.90, alarm_probability=0.99, top_k=3, persistence=2),
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
)
evaluation = EvaluationOrchestrator(pipeline, early_tolerance=0, k=3, verbose=True).evaluate([
    EvaluationCase(kuramoto_input, case_id="Kuramoto n-D", has_truth=False),
])

print("Summary:", evaluation.summary())
pd.DataFrame(evaluation.to_records())

Evaluating cases:   0%|          | 0/1 [00:00<?, ?it/s]

Summary: {'n-d': {'cases': 1, 'hit': 0, 'miss': 0, 'early_alarm': 0, 'false_alarm': 0, 'correct_rejection': 0, 'unlabelled': 1, 'detection_rate': nan, 'false_alarm_rate': nan, 'mean_delay': nan, 'median_delay': nan, 'heuristic_n': 0, 'heuristic_top1': nan, 'heuristic_top3': nan, 'heuristic_mrr': nan, 'gnn_n': 0, 'gnn_top1': nan, 'gnn_top3': nan, 'gnn_mrr': nan, 'gat_n': 0, 'gat_top1': nan, 'gat_top3': nan, 'gat_mrr': nan}}


,case_id,route,outcome,stop_reason,detected_time,true_event_time,delay,true_source,heuristic_source,heuristic_rank,gnn_source,gnn_rank,gat_source,gat_rank
0,Kuramoto n-D,n-d,unlabelled,None,300,None,None,None,8,None,None,None,None,None


Без истинного времени события нет метрики задержки или Top-k accuracy — только исполненный результат.

**Самопроверка:** почему нельзя выдумать onset только для того, чтобы получить метрику? Что изменится, если добавить второй неразмеченный n-D ряд в тот же batch? Чем этот прогон отличается от полноценного benchmark'а?